# YEDA 모델 점검 (C)

이 노트북은 `make train` 결과를 읽어 홀드아웃 지표·보정·단조 제약을 탐색하는 **읽기 전용 검토 화면**입니다. 학습이나 평가 로직을 복사하지 않고 production 모듈을 호출하며, artifact를 수정하지 않습니다. 먼저 `make data && make train`을 실행하세요.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("저장소 루트 또는 notebooks/ 에서 실행하세요.")
SRC = str(ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, learning_curve

from yeda.data.preprocess import make_split
from yeda.io_utils import load_config, resolve
from yeda.models.evaluate import (
    calibration_table,
    compute_metrics,
    expected_calibration_error,
)
from yeda.models.registry import build_model, load_bundle, verify_all_monotone_predictions

In [ ]:
model_cfg = load_config("model")
split = make_split(
    test_size=float(model_cfg["test_size"]),
    seed=int(model_cfg["seed"]),
)
bundle = load_bundle(model_cfg)
probability = bundle.predict_proba(split.X_test)
metrics = compute_metrics(
    split.y_test.to_numpy(),
    probability,
    threshold=float(bundle.threshold),
)
calibration = calibration_table(
    split.y_test.to_numpy(),
    probability,
    n_bins=int(model_cfg["evaluation"]["calibration_bins"]),
)
metrics["ece"] = expected_calibration_error(calibration)
pd.Series(metrics, name=bundle.name)

In [ ]:
comparison_path = resolve(model_cfg["evaluation"]["comparison_csv"])
comparison = pd.read_csv(comparison_path)
comparison

In [ ]:
calibration

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=int(model_cfg["seed"]))
train_sizes, train_scores, valid_scores = learning_curve(
    build_model(bundle.name, model_cfg),
    split.X_train,
    split.y_train,
    train_sizes=np.array([0.25, 0.60, 1.00]),
    scoring="average_precision",
    cv=cv,
    n_jobs=1,  # LightGBM 내부 병렬화와 중첩되지 않게 한다.
)
learning_summary = pd.DataFrame(
    {
        "n_train": train_sizes,
        "train_pr_auc": train_scores.mean(axis=1),
        "validation_pr_auc": valid_scores.mean(axis=1),
        "validation_std": valid_scores.std(axis=1),
    }
)
ax = learning_summary.plot(
    x="n_train",
    y=["train_pr_auc", "validation_pr_auc"],
    marker="o",
    figsize=(7, 4),
    title=f"{bundle.name}: PR-AUC learning curve",
)
ax.set_ylabel("PR-AUC")
ax.set_ylim(0.0, 1.02)
ax.grid(alpha=0.25)
learning_summary

In [ ]:
prediction = (probability >= bundle.threshold).astype(int)
fig, ax = plt.subplots(figsize=(4.5, 4.0))
ConfusionMatrixDisplay.from_predictions(
    split.y_test,
    prediction,
    display_labels=["Failure", "Success"],
    colorbar=False,
    ax=ax,
)
ax.set_title(f"Holdout confusion matrix (threshold={bundle.threshold:.3f})")
fig.tight_layout()
fig

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot([0, 1], [0, 1], "--", color="0.5", label="perfect calibration")
ax.plot(
    calibration["mean_predicted"],
    calibration["observed_rate"],
    marker="o",
    label=bundle.name,
)
ax.set(xlabel="Mean predicted yield", ylabel="Observed yield", xlim=(0, 1), ylim=(0, 1))
ax.set_title(f"Holdout calibration (ECE={metrics['ece']:.3f})")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig

In [ ]:
monotone_check = verify_all_monotone_predictions(
    bundle,
    n_points=int(model_cfg["verification"]["monotone_grid_points"]),
    atol=float(model_cfg["verification"]["monotone_atol"]),
)
assert monotone_check["passed"], monotone_check
pd.DataFrame(monotone_check["features"])